In [ ]:
# ⚠️  Run this cell first — sets up data and imports
# Tip: Runtime → Run all  (runs everything top to bottom automatically)

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11
})

from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA

# ── Load dataset ──────────────────────────────────────────────────────────────
ads = pd.read_csv('https://raw.githubusercontent.com/ladataanalytics/pattern-recognition-notebooks/main/data/Advertising.csv')
print(f'✓ Advertising: {ads.shape}')

print(f'  Python {sys.version.split()[0]} | pandas {pd.__version__}')
print('✓ Setup complete')


In [ ]:
# ⚠️  Run this cell first before any others
# (Tip: Runtime → Run all  OR  Shift+Enter from the top)
import pandas as pd
ads = pd.read_csv('https://raw.githubusercontent.com/ladataanalytics/pattern-recognition-notebooks/main/data/Advertising.csv')
print(f"Advertising: {ads.shape}  |  Columns: {list(ads.columns)}")
ads.head()

In [ ]:
# Summary statistics — always start here
print("=== Summary Statistics ===\n")
print(ads.describe().round(2))

In [ ]:
# Visualize: Sales vs each advertising channel
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
channels = ['tv', 'radio', 'newspaper']
colors = ['#1e5fa8', '#e85d20', '#1a7a45']

for ax, col, color in zip(axes, channels, colors):
    ax.scatter(ads[col], ads['sales'], alpha=0.5, color=color, s=30, edgecolors='white', lw=0.5)
    # Fit a simple line to show the trend
    m, b = np.polyfit(ads[col], ads['sales'], 1)
    x_line = np.linspace(ads[col].min(), ads[col].max(), 100)
    ax.plot(x_line, m*x_line + b, color='black', lw=1.5, alpha=0.7)
    ax.set_xlabel(f'{col} budget ($000s)')
    ax.set_ylabel('Sales (000 units)' if col=='tv' else '')
    ax.set_title(f'{col} vs Sales\nr = {ads[col].corr(ads["sales"]):.2f}')

plt.suptitle('Advertising Dataset — Sales vs Each Channel', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("\n📌 Observation: TV has the strongest linear relationship with sales (r=0.78).")
print("   Newspaper barely correlates. Radio is moderate.")

In [ ]:
# Simulate: demonstrate reducible vs irreducible error
np.random.seed(42)
n = 150

# True underlying function: f(X) = 3 + 2X - 0.5X²
X = np.linspace(0, 4, n)
f_true = 3 + 2*X - 0.5*X**2          # true f(X) — never observed
epsilon = np.random.normal(0, 0.8, n)  # irreducible error ε
Y = f_true + epsilon                   # observed Y = f(X) + ε

# Fit three models with increasing flexibility
models = {
    'Linear (underfit)': Pipeline([('poly', PolynomialFeatures(1)), ('lr', LinearRegression())]),
    'Quadratic (good fit)': Pipeline([('poly', PolynomialFeatures(2)), ('lr', LinearRegression())]),
    'Degree-15 (overfit)': Pipeline([('poly', PolynomialFeatures(15)), ('lr', LinearRegression())]),
}

X_2d = X.reshape(-1, 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
colors_m = ['#e85d20', '#1a7a45', '#c0392b']

for ax, (name, model), color in zip(axes, models.items(), colors_m):
    model.fit(X_2d, Y)
    Y_pred = model.predict(X_2d)
    
    train_mse = mean_squared_error(Y, Y_pred)
    irred = np.var(epsilon)
    reducible = max(0, train_mse - irred)
    
    ax.scatter(X, Y, alpha=0.3, s=20, color='#888', label='Observed data')
    ax.plot(X, f_true, 'k--', lw=1.5, label='True f(X)', alpha=0.7)
    ax.plot(X, Y_pred, color=color, lw=2.5, label=f'f̂(X)')
    ax.set_title(f'{name}\nMSE={train_mse:.2f}  |  Irred≈{irred:.2f}')
    ax.set_xlabel('X')
    if ax == axes[0]: ax.set_ylabel('Y')
    ax.legend(fontsize=8)

plt.suptitle('Reducible vs Irreducible Error — Three Model Flexibilities', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

print("\n📌 The quadratic model (middle) gets closest to the true f(X).")
print("   The degree-15 model memorizes noise — low training error but will fail on new data.")
print(f"   Irreducible error ≈ {np.var(epsilon):.2f} — no model can go below this floor.")

In [ ]:
# Show training vs test MSE as model complexity increases
np.random.seed(0)
n_train, n_test = 80, 40
X_all = np.sort(np.random.uniform(0, 4, n_train + n_test))
Y_all = 3 + 2*X_all - 0.5*X_all**2 + np.random.normal(0, 0.8, n_train + n_test)

X_tr, X_te = X_all[:n_train].reshape(-1,1), X_all[n_train:].reshape(-1,1)
Y_tr, Y_te = Y_all[:n_train], Y_all[n_train:]

# Cap at degree 10 — higher degrees explode test MSE and hide the sweet spot
degrees = range(1, 11)
train_mse, test_mse = [], []

for d in degrees:
    pipe = Pipeline([('poly', PolynomialFeatures(d)), ('lr', LinearRegression())])
    pipe.fit(X_tr, Y_tr)
    train_mse.append(mean_squared_error(Y_tr, pipe.predict(X_tr)))
    test_mse.append(mean_squared_error(Y_te, pipe.predict(X_te)))

best_degree = list(degrees)[test_mse.index(min(test_mse))]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: linear scale — shows the shape clearly
axes[0].plot(list(degrees), train_mse, 'o-', color='#1e5fa8', lw=2, label='Training MSE', markersize=6)
axes[0].plot(list(degrees), test_mse,  'o-', color='#e85d20', lw=2, label='Test MSE',     markersize=6)
axes[0].axvline(best_degree, color='#1a7a45', lw=1.5, ls='--',
                label=f'Sweet spot (degree {best_degree})')
axes[0].set_xlabel('Model Complexity (polynomial degree)')
axes[0].set_ylabel('Mean Squared Error')
axes[0].set_title('Training MSE always falls\nTest MSE has a sweet spot')
axes[0].legend()

# Right: log scale — reveals training MSE still decreasing even where it looks flat
axes[1].semilogy(list(degrees), train_mse, 'o-', color='#1e5fa8', lw=2, label='Training MSE', markersize=6)
axes[1].semilogy(list(degrees), test_mse,  'o-', color='#e85d20', lw=2, label='Test MSE',     markersize=6)
axes[1].axvline(best_degree, color='#1a7a45', lw=1.5, ls='--',
                label=f'Sweet spot (degree {best_degree})')
axes[1].set_xlabel('Model Complexity (polynomial degree)')
axes[1].set_ylabel('MSE (log scale)')
axes[1].set_title('Same data — log scale\nTraining MSE still dropping at degree 10')
axes[1].legend()

plt.suptitle('Bias-Variance Tradeoff: Training vs Test MSE', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f"\n📌 Sweet spot at degree {best_degree} — lowest test MSE")
print(f"   Degree 1:  train={train_mse[0]:.3f}  test={test_mse[0]:.3f}  ← underfitting (high bias)")
print(f"   Degree {best_degree}:  train={train_mse[best_degree-1]:.3f}  test={test_mse[best_degree-1]:.3f}  ← best generalisation")
print(f"   Degree 10: train={train_mse[-1]:.3f}  test={test_mse[-1]:.3f}  ← overfitting (high variance)")
print(f"\n   Note: training MSE keeps falling — it can always memorise training data.")
print(f"   Only test MSE tells you how well the model actually generalises.")

In [ ]:
# Visual comparison: supervised vs unsupervised
np.random.seed(7)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Supervised — labeled classes
n_each = 60
X_sup = np.vstack([
    np.random.multivariate_normal([1,1], [[0.4,0],[0,0.4]], n_each),
    np.random.multivariate_normal([3,3], [[0.4,0],[0,0.4]], n_each),
    np.random.multivariate_normal([1,3.5], [[0.3,0],[0,0.3]], n_each),
])
y_sup = np.array(['Class A']*n_each + ['Class B']*n_each + ['Class C']*n_each)
colors_sup = {'Class A':'#1e5fa8','Class B':'#e85d20','Class C':'#1a7a45'}
for cls, color in colors_sup.items():
    mask = y_sup == cls
    axes[0].scatter(X_sup[mask,0], X_sup[mask,1], color=color, label=cls, s=30, alpha=0.7)
axes[0].set_title('Supervised Learning\n(labels known — predict the class)')
axes[0].legend()
axes[0].set_xlabel('X₁'); axes[0].set_ylabel('X₂')

# Unsupervised — no labels
axes[1].scatter(X_sup[:,0], X_sup[:,1], color='#888', s=30, alpha=0.5)
axes[1].set_title('Unsupervised Learning\n(no labels — find the structure yourself)')
axes[1].set_xlabel('X₁'); axes[1].set_ylabel('X₂')

plt.suptitle('Same data, different goals', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print("📌 In supervised learning we know the answer during training.")
print("   In unsupervised learning we discover structure without any labels.")

In [ ]:
# ── Exercise workspace ──────────────────────────────────────────────────────

# Task 1: correlation
# YOUR CODE HERE

# Task 2: linear regression + train/test MSE
# YOUR CODE HERE

# Task 3: polynomial degree-3
# YOUR CODE HERE

# Task 4: written answer (as a print statement or comment)
# YOUR CODE HERE

In [ ]:
# @title 📝 Quiz — Statistical Learning
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** In Y = f(X) + ε, what does ε represent?
# @markdown **Q2:** Why does training MSE always decrease as model complexity increases?
# @markdown **Q3:** Name one example of a supervised learning problem
# @markdown **Q4:** Name one example of an unsupervised learning problem
# @markdown **Q5:** Can irreducible error be eliminated by using a more complex model? (yes/no + 1 sentence why)

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

In [ ]:
_NB_NAME_="Statistical Learning"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output box (click the copy icon in the top-left of the output box, or click ⋮ → Copy) → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    print(f"Please grade my quiz answers for the \"{_NB_TITLE}\" notebook")
    print(f"from \"Data Pattern Recognition for the Rest of Us\" (based on ISLP).")
    if GITHUB_USERNAME.strip():
        print(f"Student: @{GITHUB_USERNAME.strip()}")
    print()
    print(qa)
    print()
    print("For each question:")
    print("  1. Say CORRECT, PARTIAL, or INCORRECT")
    print("  2. Explain in 2-3 sentences why — what did I get right or wrong")
    print("  3. Give the ideal complete answer so I know exactly what was expected")
    print("  4. If I was wrong or partial, tell me the specific concept to review")
    print("     and where it appears in the notebook (e.g. Part 2, the SHAP beeswarm plot)")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - A short study plan: which questions to re-read and what to focus on")
    print()
    print("After grading, I will paste specific outputs, charts, or code blocks")
    print("from the notebook — please explain them in detail and answer follow-up questions.")


---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md) — describe the notebook name, cell number, and what went wrong.
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first to avoid duplicates.*